In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from ultralytics import YOLO
import onnxruntime as ort
import os

model_path = 'pothole_detector_v1.onnx'


try:
    best_model = YOLO(model_path)
    print(f"Successfully loaded model from {model_path} using YOLO")
except Exception as e:
    print(f"Error loading with YOLO: {e}")

In [ ]:
# Define the video path
video_path = '/kaggle/working/sample_video.mp4'

# Define font, scale, colors, and position for the annotation
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1
text_position = (40, 80)
font_color = (255, 255, 255)    # White color for text
background_color = (0, 0, 255)  # Red background for text

# Initialize a deque with fixed length for averaging the last 10 percentage damages
damage_deque = deque(maxlen=10)

# Open the video
cap = cv2.VideoCapture(video_path)

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('road_damage_assessment.avi', fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

# Read until video is completed
while cap.isOpened():
     # Capture frame-by-frame
    ret, frame = cap.read()
    if ret:
        # Perform inference on the frame
        results = best_model.predict(source=frame, imgsz=640, conf=0.25)
        processed_frame = results[0].plot(boxes=False)
        
        # Initializes percentage_damage to 0
        percentage_damage = 0 
        
        # If masks are available, calculate total damage area and percentage
        if results[0].masks is not None:
            total_area = 0
            masks = results[0].masks.data.cpu().numpy()
            image_area = frame.shape[0] * frame.shape[1]  # total number of pixels in the image
            for mask in masks:
                binary_mask = (mask > 0).astype(np.uint8) * 255
                contour, _ = cv2.findContours(binary_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
                total_area += cv2.contourArea(contour[0])
            
            percentage_damage = (total_area / image_area) * 100

        # Calculate and update the percentage damage
        damage_deque.append(percentage_damage)
        smoothed_percentage_damage = sum(damage_deque) / len(damage_deque)
            
        # Draw a thick line for text background
        cv2.line(processed_frame, (text_position[0], text_position[1] - 10),
                 (text_position[0] + 350, text_position[1] - 10), background_color, 40)
        
        # Annotate the frame with the percentage of damage
        cv2.putText(processed_frame, f'Road Damage: {smoothed_percentage_damage:.2f}%', text_position, font, font_scale, font_color, 2, cv2.LINE_AA)         
    
        # Write the processed frame to the output video
        out.write(processed_frame)
        
        # Uncomment the following 3 lines if running this code on a local machine to view the real-time processing results
        # cv2.imshow('Road Damage Assessment', processed_frame) # Display the processed frame
        # if cv2.waitKey(1) & 0xFF == ord('q'): # Press Q on keyboard to exit the loop
        #     break 
    else:
        break

# Release the video capture and video write objects
cap.release()
out.release()

# Close all the frames
# cv2.destroyAllWindows()

In [ ]:
# Convert the .avi video generated by our traffic density estimation app to .mp4 format for compatibility with notebook display
!ffmpeg -y -loglevel panic -i /kaggle/working/road_damage_assessment.avi road_damage_assessment.mp4

# Embed and display the processed sample video within the notebook
Video("road_damage_assessment.mp4", embed=True, width=960)

In [ ]:
from collections import defaultdict, deque
import os
import cv2
import numpy as np


def estimate_pothole_depth(image, binary_mask, contour):
    """
    Estimates the depth of a pothole based on shadow analysis in the pothole region.
    
    Args:
        image: Input image (BGR format)
        binary_mask: Binary mask of the pothole
        contour: Contour of the pothole
        
    Returns:
        depth_score: Estimated depth score (0-1)
    """
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Create mask from contour for precise region analysis
    mask = np.zeros_like(gray_image)
    cv2.drawContours(mask, [contour], 0, 255, -1)
    
    # Extract only the pothole region using the mask
    pothole_region = cv2.bitwise_and(gray_image, gray_image, mask=mask)
    
    # Get pixel values excluding zeros (background)
    pixel_values = pothole_region[pothole_region > 0]
    
    if len(pixel_values) == 0:
        return 0.0  # No valid pixels
    
    # Calculate statistics of the pothole region
    mean_value = np.mean(pixel_values)
    min_value = np.min(pixel_values)
    
    # Calculate the depth score based on darkness and contrast
    # Darker regions indicate deeper potholes
    # Normalize to 0-1 range where 1 is deepest
    darkness_score = 1 - (mean_value / 255.0)
    
    # Calculate contrast within the pothole (higher contrast often means deeper)
    if len(pixel_values) > 1:
        std_dev = np.std(pixel_values)
        contrast_score = min(std_dev / 50.0, 1.0)  # Normalize, cap at 1.0
    else:
        contrast_score = 0.0
    
    # Combined score with more weight on darkness
    depth_score = (0.7 * darkness_score) + (0.3 * contrast_score)
    
    # Ensure it's in 0-1 range
    depth_score = max(0.0, min(1.0, depth_score))
    
    return depth_score


def get_individual_pothole_priority(area_ratio, depth_score):
    """
    Determines the priority of an individual pothole based on size and depth.
    
    Args:
        area_ratio: Ratio of pothole area to image area
        depth_score: Estimated depth score (0-1)
        
    Returns:
        priority: String priority level ('High', 'Medium', or 'Low')
        color: BGR color tuple for visualization
    """
    # Calculate combined score weighted by area and depth
    # Area is more important for overall road damage assessment
    combined_score = (0.6 * area_ratio * 100) + (0.4 * depth_score)
    
    # Determine priority based on combined score
    if combined_score > 0.4 or (area_ratio > 0.01 and depth_score > 0.6):
        priority = 'High'
        color = (0, 0, 255)  # Red (BGR)
    elif combined_score > 0.2 or (area_ratio > 0.005 and depth_score > 0.4):
        priority = 'Medium'
        color = (0, 165, 255)  # Orange (BGR)
    else:
        priority = 'Low'
        color = (0, 255, 0)  # Green (BGR)
    
    return priority, color


def determine_road_priority(potholes_list, proximity_threshold, image_shape):
    """
    Determines the overall road priority based on pothole count, proximity, and severity.
    
    Args:
        potholes_list: List of detected potholes with priority information
        proximity_threshold: Maximum distance to consider potholes as clustered
        image_shape: Shape of the input image
        
    Returns:
        road_priority: Overall road priority ('High', 'Medium', or 'Low')
        road_color: BGR color tuple for visualization
        clusters: List of clusters, where each cluster is a list of pothole indices
    """
    # If no potholes, return low priority
    if not potholes_list:
        return 'Low', (0, 255, 0), []
    
    # Count high and medium priority potholes
    high_priority_count = sum(1 for p in potholes_list if p['priority'] == 'High')
    medium_priority_count = sum(1 for p in potholes_list if p['priority'] == 'Medium')
    
    # Find clusters of potholes (potholes in proximity indicate concentrated damage)
    clusters = []
    processed = set()
    
    for i, pothole1 in enumerate(potholes_list):
        if i in processed:
            continue
        
        cluster = [i]
        processed.add(i)
        
        for j, pothole2 in enumerate(potholes_list):
            if j in processed or i == j:
                continue
            
            # Calculate distance between potholes
            pos1 = pothole1['position']
            pos2 = pothole2['position']
            distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
            
            # If close enough, add to cluster
            if distance < proximity_threshold:
                cluster.append(j)
                processed.add(j)
        
        clusters.append(cluster)
    
    # Calculate total damaged area (as percentage of road)
    total_area_ratio = sum(p['area_ratio'] for p in potholes_list)
    
    # Calculate area of largest cluster as a percentage of image area
    largest_cluster_area = 0
    if clusters:
        for cluster in clusters:
            if len(cluster) > 1:  # Only consider clusters with multiple potholes
                cluster_points = np.array([potholes_list[idx]['position'] for idx in cluster])
                hull = cv2.convexHull(cluster_points.reshape(-1, 1, 2))
                cluster_area = cv2.contourArea(hull) / (image_shape[0] * image_shape[1])
                largest_cluster_area = max(largest_cluster_area, cluster_area)
    
    # Determine road priority using multiple factors
    if (high_priority_count >= 2 or 
        (high_priority_count >= 1 and medium_priority_count >= 2) or
        total_area_ratio > 0.05 or
        largest_cluster_area > 0.03 or
        len([c for c in clusters if len(c) >= 3]) >= 1):  # Cluster with 3+ potholes
        road_priority = 'High'
        road_color = (0, 0, 255)  # Red (BGR)
    elif (high_priority_count >= 1 or 
          medium_priority_count >= 2 or
          total_area_ratio > 0.02 or
          largest_cluster_area > 0.015 or
          len([c for c in clusters if len(c) >= 2]) >= 1):  # Cluster with 2+ potholes
        road_priority = 'Medium'
        road_color = (0, 165, 255)  # Orange (BGR)
    else:
        road_priority = 'Low'
        road_color = (0, 255, 0)  # Green (BGR)
    
    return road_priority, road_color, clusters


# Add required imports at the top of your file


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt



# Example usage for a single image
def process_road_image_example(image_path):
    try:
        # Process the image to assess road priority
        annotated_image, road_info = assess_road_priority(
            image_path, 
            conf_threshold=0.25,
            proximity_threshold=150,  # Adjust based on your image scale
            model=model
        )
        
        # Display the results
        plt.figure(figsize=(12, 8))
        
        # Convert BGR to RGB for matplotlib
        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
        plt.imshow(annotated_image_rgb)
        plt.title(f"Road Assessment: {road_info['road_priority']} Priority\n"
                f"Total Potholes: {road_info['total_potholes']}, Clusters: {road_info['pothole_clusters']}")
        plt.axis('off')
        plt.savefig('road_assessment_result.png')
        plt.show()
        
        print("\nRoad Priority Assessment Summary:")
        print(f"Road Priority: {road_info['road_priority']}")
        print(f"Total Potholes: {road_info['total_potholes']}")
        print(f"Pothole Clusters: {road_info['pothole_clusters']}")
        print("\nIndividual Pothole Priorities:")
        print(f"  High: {road_info['individual_priorities']['High']}")
        print(f"  Medium: {road_info['individual_priorities']['Medium']}")
        print(f"  Low: {road_info['individual_priorities']['Low']}")
        
        return annotated_image, road_info
    
    except Exception as e:
        print(f"Error processing image: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Example usage for video processing
def process_road_video_example(video_path):
    try:
        # Process the video
        output_path, road_summary = process_video_for_road_priority(
            video_path, 
            conf_threshold=0.25,
            proximity_threshold=150,  # Adjust based on your video scale
            model=model
        )
        
        print("\nVideo Processing Complete!")
        print(f"Processed video saved to: {output_path}")
        print("\nRoad Assessment Summary:")
        print(f"Total frames processed: {road_summary['total_frames']}")
        print(f"Most common road priority: {road_summary['most_common_road_priority']}")
        print("\nPriority Distribution:")
        print(f"  High: {road_summary['priority_distribution']['High']} frames ({road_summary['high_priority_percentage']:.2f}%)")
        print(f"  Medium: {road_summary['priority_distribution']['Medium']} frames")
        print(f"  Low: {road_summary['priority_distribution']['Low']} frames")
        
        return output_path, road_summary
    
    except Exception as e:
        print(f"Error processing video: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Example usage:
# annotated_image, road_info = process_road_image_example('/path/to/your/image.jpg')
# output_video, road_summary = process_road_video_example('/path/to/your/video.mp4')

In [ ]:
process_road_image_example("/kaggle/input/testpothole/bad-road-cracked.webp")

In [ ]:
process_road_video_example("road_damage_assessment.mp4")

In [ ]:
import cv2
import numpy as np
import json
from collections import defaultdict, Counter
import os

def estimate_pothole_depth(image, binary_mask, contour):
    """
    Estimates the depth of a pothole based on shadow analysis in the pothole region.
    
    Args:
        image: Input image (BGR format)
        binary_mask: Binary mask of the pothole
        contour: Contour of the pothole
        
    Returns:
        depth_score: Estimated depth score (0-1)
    """
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Create mask from contour for precise region analysis
    mask = np.zeros_like(gray_image)
    cv2.drawContours(mask, [contour], 0, 255, -1)
    
    # Extract only the pothole region using the mask
    pothole_region = cv2.bitwise_and(gray_image, gray_image, mask=mask)
    
    # Get pixel values excluding zeros (background)
    pixel_values = pothole_region[pothole_region > 0]
    
    if len(pixel_values) == 0:
        return 0.0  # No valid pixels
    
    # Calculate statistics of the pothole region
    mean_value = np.mean(pixel_values)
    min_value = np.min(pixel_values)
    
    # Calculate the depth score based on darkness and contrast
    # Darker regions indicate deeper potholes
    # Normalize to 0-1 range where 1 is deepest
    darkness_score = 1 - (mean_value / 255.0)
    
    # Calculate contrast within the pothole (higher contrast often means deeper)
    if len(pixel_values) > 1:
        std_dev = np.std(pixel_values)
        contrast_score = min(std_dev / 50.0, 1.0)  # Normalize, cap at 1.0
    else:
        contrast_score = 0.0
    
    # Combined score with more weight on darkness
    depth_score = (0.7 * darkness_score) + (0.3 * contrast_score)
    
    # Ensure it's in 0-1 range
    depth_score = max(0.0, min(1.0, depth_score))
    
    return depth_score


def get_individual_pothole_priority(area_ratio, depth_score):
    """
    Determines the priority of an individual pothole based on size and depth.
    
    Args:
        area_ratio: Ratio of pothole area to image area
        depth_score: Estimated depth score (0-1)
        
    Returns:
        priority: String priority level ('High', 'Medium', or 'Low')
        color: BGR color tuple for visualization
    """
    # Calculate combined score weighted by area and depth
    # Area is more important for overall road damage assessment
    combined_score = (0.6 * area_ratio * 100) + (0.4 * depth_score)
    
    # Determine priority based on combined score
    if combined_score > 0.4 or (area_ratio > 0.01 and depth_score > 0.6):
        priority = 'High'
        color = (0, 0, 255)  # Red (BGR)
    elif combined_score > 0.2 or (area_ratio > 0.005 and depth_score > 0.4):
        priority = 'Medium'
        color = (0, 165, 255)  # Orange (BGR)
    else:
        priority = 'Low'
        color = (0, 255, 0)  # Green (BGR)
    
    return priority, color


def determine_road_priority(potholes_list, proximity_threshold, image_shape):
    """
    Determines the overall road priority based on pothole count, proximity, and severity.
    
    Args:
        potholes_list: List of detected potholes with priority information
        proximity_threshold: Maximum distance to consider potholes as clustered
        image_shape: Shape of the input image
        
    Returns:
        road_priority: Overall road priority ('High', 'Medium', or 'Low')
        road_color: BGR color tuple for visualization
        clusters: List of clusters, where each cluster is a list of pothole indices
    """
    # If no potholes, return low priority
    if not potholes_list:
        return 'Low', (0, 255, 0), []
    
    # Count high and medium priority potholes
    high_priority_count = sum(1 for p in potholes_list if p['priority'] == 'High')
    medium_priority_count = sum(1 for p in potholes_list if p['priority'] == 'Medium')
    
    # Find clusters of potholes (potholes in proximity indicate concentrated damage)
    clusters = []
    processed = set()
    
    for i, pothole1 in enumerate(potholes_list):
        if i in processed:
            continue
        
        cluster = [i]
        processed.add(i)
        
        for j, pothole2 in enumerate(potholes_list):
            if j in processed or i == j:
                continue
            
            # Calculate distance between potholes
            pos1 = pothole1['position']
            pos2 = pothole2['position']
            distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
            
            # If close enough, add to cluster
            if distance < proximity_threshold:
                cluster.append(j)
                processed.add(j)
        
        clusters.append(cluster)
    
    # Calculate total damaged area (as percentage of road)
    total_area_ratio = sum(p['area_ratio'] for p in potholes_list)
    
    # Calculate area of largest cluster as a percentage of image area
    largest_cluster_area = 0
    if clusters:
        for cluster in clusters:
            if len(cluster) > 1:  # Only consider clusters with multiple potholes
                cluster_points = np.array([potholes_list[idx]['position'] for idx in cluster])
                hull = cv2.convexHull(cluster_points.reshape(-1, 1, 2))
                cluster_area = cv2.contourArea(hull) / (image_shape[0] * image_shape[1])
                largest_cluster_area = max(largest_cluster_area, cluster_area)
    
    # Determine road priority using multiple factors
    if (high_priority_count >= 2 or 
        (high_priority_count >= 1 and medium_priority_count >= 2) or
        total_area_ratio > 0.05 or
        largest_cluster_area > 0.03 or
        len([c for c in clusters if len(c) >= 3]) >= 1):  # Cluster with 3+ potholes
        road_priority = 'High'
        road_color = (0, 0, 255)  # Red (BGR)
    elif (high_priority_count >= 1 or 
          medium_priority_count >= 2 or
          total_area_ratio > 0.02 or
          largest_cluster_area > 0.015 or
          len([c for c in clusters if len(c) >= 2]) >= 1):  # Cluster with 2+ potholes
        road_priority = 'Medium'
        road_color = (0, 165, 255)  # Orange (BGR)
    else:
        road_priority = 'Low'
        road_color = (0, 255, 0)  # Green (BGR)
    
    return road_priority, road_color, clusters


def assess_road_image(image_path, model, conf_threshold=0.25, proximity_threshold=150):
    """
    Assesses a road image and detects potholes, returning both JSON data and annotated image
    
    Args:
        image_path: Path to the image
        model: YOLOv8 model for pothole detection
        conf_threshold: Confidence threshold for pothole detection
        proximity_threshold: Threshold for pothole clustering
        
    Returns:
        json_output: JSON string with road assessment data
        annotated_image: Image with pothole detections visualized
    """
    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not read image at {image_path}")
    
    # Create a copy for annotation
    annotated_image = image.copy()
    
    # Get image dimensions
    image_height, image_width = image.shape[:2]
    image_area = image_height * image_width
    
    # Detect potholes using the model
    results = model(image, conf=conf_threshold)
    detections = results[0].boxes
    
    # Process pothole detections
    potholes_list = []
    for i, det in enumerate(detections):
        # Get bounding box coordinates
        x1, y1, x2, y2 = map(int, det.xyxy[0])
        confidence = float(det.conf[0])
        
        # Calculate pothole area
        pothole_area = (x2 - x1) * (y2 - y1)
        area_ratio = pothole_area / image_area
        
        # Create contour for depth estimation
        contour = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.int32).reshape((-1, 1, 2))
        
        # Create binary mask for the pothole region
        binary_mask = np.zeros((image_height, image_width), dtype=np.uint8)
        cv2.drawContours(binary_mask, [contour], 0, 255, -1)
        
        # Estimate pothole depth
        depth_score = estimate_pothole_depth(image, binary_mask, contour)
        
        # Determine individual pothole priority
        priority, color = get_individual_pothole_priority(area_ratio, depth_score)
        
        # Calculate pothole center position
        center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
        
        # Store pothole information
        potholes_list.append({
            'id': i,
            'position': (center_x, center_y),
            'bbox': [x1, y1, x2, y2],
            'area_ratio': area_ratio,
            'depth_score': depth_score,
            'priority': priority,
            'confidence': confidence
        })
        
        # Annotate the image
        cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)
        
        # Add priority label
        label = f"{priority} ({confidence:.2f})"
        cv2.putText(annotated_image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    # Determine overall road priority
    road_priority, road_color, clusters = determine_road_priority(
        potholes_list, proximity_threshold, (image_height, image_width))
    
    # Highlight clusters if any exist
    for cluster_idx, cluster in enumerate(clusters):
        if len(cluster) > 1:  # Only highlight clusters with multiple potholes
            # Get all points in this cluster
            cluster_points = np.array([potholes_list[idx]['position'] for idx in cluster])
            
            # Draw convex hull around the cluster
            hull = cv2.convexHull(cluster_points.reshape(-1, 1, 2))
            cv2.polylines(annotated_image, [hull], True, (255, 0, 255), 2)
            
            # Label the cluster
            centroid = np.mean(cluster_points, axis=0, dtype=np.int32)
            cv2.putText(annotated_image, f"Cluster {cluster_idx+1}", 
                        tuple(centroid), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)
    
    # Add road priority label at the top of the image
    title = f"Road Priority: {road_priority}"
    cv2.putText(annotated_image, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, road_color, 3)
    
    # Count individual pothole priorities
    priority_counts = {'High': 0, 'Medium': 0, 'Low': 0}
    for pothole in potholes_list:
        priority_counts[pothole['priority']] += 1
    
    # Create JSON output
    assessment_data = {
        "assessment_type": "image",
        "source": os.path.basename(image_path),
        "road_priority": road_priority,
        "total_potholes": len(potholes_list),
        "priority_distribution": priority_counts,
        "cluster_count": len([c for c in clusters if len(c) > 1]),
        "potholes": [
            {
                "id": p["id"],
                "bbox": p["bbox"],
                "priority": p["priority"],
                "depth_score": round(p["depth_score"], 2),
                "confidence": round(p["confidence"], 2)
            } for p in potholes_list
        ]
    }
    
    json_output = json.dumps(assessment_data, indent=2)
    
    return json_output, annotated_image


def assess_road_video(video_path, model, output_path=None, conf_threshold=0.25, 
                     proximity_threshold=150, process_every_n_frames=5):
    """
    Processes a video to detect potholes and assess road priority
    
    Args:
        video_path: Path to the input video
        model: YOLOv8 model for pothole detection
        output_path: Path for the output video (if None, auto-generated)
        conf_threshold: Confidence threshold for pothole detection
        proximity_threshold: Threshold for pothole clustering
        process_every_n_frames: Process every n frames (to improve speed)
        
    Returns:
        json_output: JSON string with road assessment data
        output_video_path: Path to the annotated output video
    """
    # Create output path if not provided
    if output_path is None:
        base_name = os.path.splitext(os.path.basename(video_path))[0]
        output_path = f"{base_name}_assessed.mp4"
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video at {video_path}")
    
    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    
    # Initialize variables
    frame_priorities = []
    all_potholes = []
    frame_number = 0
    processed_frames = 0
    
    # Process the video
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Create a copy of the frame for potential skipped frames
        output_frame = frame.copy()
        
        # Process every n frames to improve speed
        if frame_number % process_every_n_frames == 0:
            # Get frame dimensions
            frame_area = frame_height * frame_width
            
            # Detect potholes using the model
            results = model(frame, conf=conf_threshold)
            detections = results[0].boxes
            
            # Process pothole detections
            potholes_list = []
            for i, det in enumerate(detections):
                # Get bounding box coordinates
                x1, y1, x2, y2 = map(int, det.xyxy[0])
                confidence = float(det.conf[0])
                
                # Calculate pothole area
                pothole_area = (x2 - x1) * (y2 - y1)
                area_ratio = pothole_area / frame_area
                
                # Create contour for depth estimation
                contour = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.int32).reshape((-1, 1, 2))
                
                # Create binary mask for the pothole region
                binary_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
                cv2.drawContours(binary_mask, [contour], 0, 255, -1)
                
                # Estimate pothole depth
                depth_score = estimate_pothole_depth(frame, binary_mask, contour)
                
                # Determine individual pothole priority
                priority, color = get_individual_pothole_priority(area_ratio, depth_score)
                
                # Calculate pothole center position
                center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                
                # Store pothole information
                pothole_info = {
                    'id': i,
                    'position': (center_x, center_y),
                    'bbox': [x1, y1, x2, y2],
                    'area_ratio': area_ratio,
                    'depth_score': depth_score,
                    'priority': priority,
                    'confidence': confidence,
                    'frame': frame_number
                }
                potholes_list.append(pothole_info)
                all_potholes.append(pothole_info)
                
                # Annotate the frame
                cv2.rectangle(output_frame, (x1, y1), (x2, y2), color, 2)
                
                # Add priority label
                label = f"{priority} ({confidence:.2f})"
                cv2.putText(output_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Determine overall road priority
            road_priority, road_color, clusters = determine_road_priority(
                potholes_list, proximity_threshold, (frame_height, frame_width))
            
            # Highlight clusters if any exist
            for cluster_idx, cluster in enumerate(clusters):
                if len(cluster) > 1:  # Only highlight clusters with multiple potholes
                    # Get all points in this cluster
                    cluster_points = np.array([potholes_list[idx]['position'] for idx in cluster])
                    
                    # Draw convex hull around the cluster
                    hull = cv2.convexHull(cluster_points.reshape(-1, 1, 2))
                    cv2.polylines(output_frame, [hull], True, (255, 0, 255), 2)
            
            # Store frame priority
            frame_priorities.append(road_priority)
            processed_frames += 1
            
            # Add frame number and road priority text
            cv2.putText(output_frame, f"Frame: {frame_number}", (10, 25), 
                      cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            cv2.putText(output_frame, f"Road Priority: {road_priority}", (10, 50), 
                      cv2.FONT_HERSHEY_SIMPLEX, 0.6, road_color, 2)
        
        # Write the frame to output video
        out.write(output_frame)
        frame_number += 1
    
    # Release resources
    cap.release()
    out.release()
    
    # Calculate summary statistics
    priority_distribution = Counter(frame_priorities)
    most_common_priority = priority_distribution.most_common(1)[0][0] if frame_priorities else 'Unknown'
    
    # Count individual pothole priorities across all frames
    pothole_priorities = [p['priority'] for p in all_potholes]
    pothole_priority_counts = Counter(pothole_priorities)
    
    # Calculate percentages
    high_percentage = (priority_distribution.get('High', 0) / processed_frames) * 100 if processed_frames > 0 else 0
    medium_percentage = (priority_distribution.get('Medium', 0) / processed_frames) * 100 if processed_frames > 0 else 0
    low_percentage = (priority_distribution.get('Low', 0) / processed_frames) * 100 if processed_frames > 0 else 0
    
    # Create JSON output
    assessment_data = {
        "assessment_type": "video",
        "source": os.path.basename(video_path),
        "output_video": os.path.basename(output_path),
        "total_frames": frame_count,
        "processed_frames": processed_frames,
        "most_common_road_priority": most_common_priority,
        "frame_priority_distribution": {
            "High": priority_distribution.get('High', 0),
            "Medium": priority_distribution.get('Medium', 0),
            "Low": priority_distribution.get('Low', 0)
        },
        "priority_percentages": {
            "High": round(high_percentage, 1),
            "Medium": round(medium_percentage, 1),
            "Low": round(low_percentage, 1)
        },
        "total_potholes_detected": len(all_potholes),
        "pothole_priority_distribution": {
            "High": pothole_priority_counts.get('High', 0),
            "Medium": pothole_priority_counts.get('Medium', 0),
            "Low": pothole_priority_counts.get('Low', 0)
        },
        "road_repair_recommendation": most_common_priority
    }
    
    json_output = json.dumps(assessment_data, indent=2)
    
    return json_output, output_path


# Example usage
def process_road_image_example(image_path, model):
    """
    Example function to process a single road image and visualize results
    """
    try:
        # Process the image
        json_output, annotated_image = assess_road_image(
            image_path, 
            model=model,
            conf_threshold=0.25,
            proximity_threshold=150
        )
        
        # Save the annotated image
        output_image_path = "annotated_" + os.path.basename(image_path)
        cv2.imwrite(output_image_path, annotated_image)
        
        # Print JSON output
        print(json_output)
        
        return json_output, output_image_path
    
    except Exception as e:
        print(f"Error processing image: {e}")
        import traceback
        traceback.print_exc()
        return None, None


def process_road_video_example(video_path, model):
    """
    Example function to process a road video and visualize results
    """
    try:
        # Process the video
        json_output, output_video_path = assess_road_video(
            video_path, 
            model=model,
            conf_threshold=0.25,
            proximity_threshold=150,
            process_every_n_frames=5
        )
        
        # Print JSON output
        print(json_output)
        
        return json_output, output_video_path
    
    except Exception as e:
        print(f"Error processing video: {e}")
        import traceback
        traceback.print_exc()
        return None, None


# Main execution block - for testing
if __name__ == "__main__":
    try:
        
        model = best_model
        print("Model loaded successfully!")
        
        # Process image - update with your actual image path
        img_path = "/kaggle/input/testpothole/bad-road-cracked.webp"  # Replace with your actual image path
        if os.path.exists(img_path):
            print(f"Processing image: {img_path}")
            json_output, output_image_path = process_road_image_example(img_path, model)
            if output_image_path:
                print(f"Annotated image saved to: {output_image_path}")
        else:
            print(f"Image file not found: {img_path}")
            print("Please specify a valid image path to process")
        
        # Process video - update with your actual video path
        video_path = "road_damage_assessment.mp4"  # Replace with your actual video path
        if os.path.exists(video_path):
            print(f"Processing video: {video_path}")
            json_output, output_video_path = process_road_video_example(video_path, model)
            if output_video_path:
                print(f"Annotated video saved to: {output_video_path}")
        else:
            print(f"Video file not found: {video_path}")
            print("Please specify a valid video path to process")
            
    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()